# modkit DMR multi-sample tutorial

This notebook is for multi-sample DMR analyses using native `modkit dmr` wrappers in `dimelo.workflows`.


In [ ]:
import importlib
from pathlib import Path
from dimelo import run_modkit, workflows

importlib.reload(run_modkit)
importlib.reload(workflows)
run_modkit._get_modkit_capabilities_cached.cache_clear()
caps = run_modkit.get_modkit_capabilities()
print(caps.version_raw)


## Example inputs

Set these to your sample pileup bedMethyl outputs (`*.bed.gz`) and reference FASTA.


In [ ]:
ref_genome = Path("dimelo/test/output/chm13.draft_v1.0.fasta")
regions_bed = Path("dimelo/test/data/ctcf_demo_peak.bed")
sample_pileups = {
    "sample_a": Path("path/to/sample_a.pileup.sorted.bed.gz"),
    "sample_b": Path("path/to/sample_b.pileup.sorted.bed.gz"),
    # "sample_c": Path("path/to/sample_c.pileup.sorted.bed.gz"),
}


## Pairwise DMR + HMM segmentation

Use this when you want single-site beta-binomial statistics plus segmented DMR blocks.


In [ ]:
pair_result = workflows.modkit_dmr_pair_workflow(
    control_bed_methyl=sample_pileups["sample_a"],
    experiment_bed_methyl=sample_pileups["sample_b"],
    ref_genome=ref_genome,
    out_path="artifacts/dmr/sample_a_vs_sample_b.sites.bed",
    segment_path="artifacts/dmr/sample_a_vs_sample_b.segments.bed",
    bases=["A"],
    threads=4,
)
print(pair_result.output_path)
print(pair_result.segment_path)
display(pair_result.high_confidence_sites.head())


## Multi-sample regional DMR

`modkit dmr multi` requires a region BED and emits all-vs-all pair BED outputs.


In [ ]:
multi_result = workflows.modkit_dmr_multi_workflow(
    samples=sample_pileups,
    regions_bed=regions_bed,
    ref_genome=ref_genome,
    out_dir="artifacts/dmr/multi",
    bases=["A"],
    threads=4,
)
display(multi_result.pair_files)
